In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ======================== configuration ========================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
SIM_RUN = Path("../outputs/simulation")
DML_RUN = Path("../outputs/dml")
SHAP_BASE = Path("../outputs/shap")
OUT = Path("../outputs/tables")
COAST_KM = 10.0
POP_YEAR = 2020
# ===============================================================

OUT.mkdir(parents=True, exist_ok=True)
TEX = []


def require(ok, msg):
    if not ok:
        raise ValueError(msg)


def minus(text):
    return text.replace("-", "$-$")


# ------------------------------------------------------------
# Table S1  City characteristics
# ------------------------------------------------------------
DISPLAY = {
    "Abidjan_IvoryCoast": ("Abidjan", "Ivory Coast"), "AddisAbaba_Ethiopia": ("Addis Ababa", "Ethiopia"),
    "Alexandria_Egypt": ("Alexandria", "Egypt"), "Bamako_Mali": ("Bamako", "Mali"),
    "Cairo_Egypt": ("Cairo", "Egypt"), "DaresSalaam_Tanzania": ("Dar es Salaam", "Tanzania"),
    "Johannesburg_SouthAfrica": ("Johannesburg", "South Africa"), "Kinshasa_DRC": ("Kinshasa", "DRC"),
    "Lagos_Nigeria": ("Lagos", "Nigeria"), "Nairobi_Kenya": ("Nairobi", "Kenya"),
    "Bangkok_Thailand": ("Bangkok", "Thailand"), "HoChiMinh_Vietnam": ("Ho Chi Minh City", "Vietnam"),
    "Jakarta_Indonesia": ("Jakarta", "Indonesia"), "KualaLumpur_Malaysia": ("Kuala Lumpur", "Malaysia"),
    "Manila_Philippines": ("Manila", "Philippines"), "PhnomPenh_Cambodia": ("Phnom Penh", "Cambodia"),
    "Bangalore_India": ("Bangalore", "India"), "Chennai_India": ("Chennai", "India"),
    "Colombo_SriLanka": ("Colombo", "Sri Lanka"), "Delhi_India": ("Delhi", "India"),
    "Dhaka_Bangladesh": ("Dhaka", "Bangladesh"), "Hyderabad_India": ("Hyderabad", "India"),
    "Karachi_Pakistan": ("Karachi", "Pakistan"), "Lahore_Pakistan": ("Lahore", "Pakistan"),
    "Mumbai_India": ("Mumbai", "India"), "BeloHorizonte_Brazil": ("Belo Horizonte", "Brazil"),
    "Bogota_Colombia": ("Bogota", "Colombia"), "BuenosAires_Argentina": ("Buenos Aires", "Argentina"),
    "Guayaquil_Ecuador": ("Guayaquil", "Ecuador"), "Lima_Peru": ("Lima", "Peru"),
    "Manaus_Brazil": ("Manaus", "Brazil"), "MexicoCity_Mexico": ("Mexico City", "Mexico"),
    "Santiago_Chile": ("Santiago", "Chile"), "SantoDomingo_DominicanRepublic": ("Santo Domingo", "Dominican Rep."),
    "SaoPaulo_Brazil": ("Sao Paulo", "Brazil"),
}
LATEX_NAME = {"Bogota": r"Bogot\'a", "Sao Paulo": r"S\~ao Paulo"}
REGIONS = [("Africa", "AF"), ("Southeast Asia", "SEA"), ("South Asia", "SA"), ("Latin America", "LA")]
CLIMATE = {"Tropical (A)": "Tropical", "Arid (B)": "Arid", "Temperate/Highland (C)": "Temperate"}

panel = pd.read_csv(PANEL, low_memory=False)
require(len(panel) == 5216 and panel[["city", "cell_id", "year"]].drop_duplicates().shape[0] == 5216, "unexpected panel size")
require(set(panel.city) == set(DISPLAY), "city names differ from DISPLAY")
require(set(panel.region) == {r for r, _ in REGIONS}, "unexpected region labels")
require(set(panel.koppen_broad) == set(CLIMATE), "unexpected climate labels")
analysed = panel.sample_dml_controls.astype(str).str.lower().eq("true")

one_year = panel[panel.year.eq(POP_YEAR)]
require(one_year.groupby(["city", "cell_id"]).size().eq(1).all(), "duplicated cells in the population year")
s1 = one_year.groupby("city").agg(
    region=("region", "first"), climate=("koppen_broad", "first"), cells=("cell_id", "nunique"),
    population=("pop_study", "sum"), study_area_km2=("study_area_km2", "sum"), elevation_m=("dem_mean_m", "mean"),
    min_coast_km=("coast_dist_km", "min"))
s1["analysed_cells"] = panel[analysed].groupby("city").cell_id.nunique().reindex(s1.index).fillna(0).astype(int)
s1["coastal"] = s1.min_coast_km.le(COAST_KM)
s1["population_million"] = s1.population / 1e6
s1["density_per_km2"] = s1.population / s1.study_area_km2
s1["city_name"] = [DISPLAY[c][0] for c in s1.index]
s1["country"] = [DISPLAY[c][1] for c in s1.index]
require(s1.cells.sum() == 652 and s1.analysed_cells.sum() == 572, "cell counts differ from 652 / 572")

TEX += ["% ===== Table S1: City & Country & Reg. & Climate & Coast. & Pop. & Density & DEM & Study area & Cells & Analysed ====="]
rows = []
for region, code in REGIONS:
    TEX.append(r"\multicolumn{11}{l}{\textit{%s}} \\" % region)
    sub = s1[s1.region.eq(region)].sort_values("city_name")
    for _, r in sub.iterrows():
        rows.append({"city": r.city_name, "country": r.country, "region": code, "climate": CLIMATE[r.climate],
                     "coastal": "Yes" if r.coastal else "No", "population_million": round(r.population_million, 2),
                     "density_per_km2": round(r.density_per_km2), "elevation_m": round(r.elevation_m, 1),
                     "study_area_km2": round(r.study_area_km2, 1), "cells": int(r.cells), "analysed_cells": int(r.analysed_cells)})
        TEX.append(f"{LATEX_NAME.get(r.city_name, r.city_name)} & {r.country} & {code} & {CLIMATE[r.climate]} & "
                   f"{'Yes' if r.coastal else 'No'} & {r.population_million:.2f} & {r.density_per_km2:,.0f} & "
                   f"{r.elevation_m:,.1f} & {r.study_area_km2:,.1f} & {int(r.cells)} & {int(r.analysed_cells)} \\\\")
    TEX.append(r"\addlinespace")
table_s1 = pd.DataFrame(rows)
table_s1.to_csv(OUT / "table_S1_city_characteristics.csv", index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# Table S2  Siting rules by heatwave type (main setting)
# ------------------------------------------------------------
RULES = [("population", "Population"), ("density", "Building density"), ("height", "Building height"),
         ("random", "Random"), ("hwd_hist", "Historical HWD"), ("hwd_foresight", "Concurrent HWD"), ("oracle", "Oracle")]
TYPES = [("any", "Total"), ("day", "Daytime-only"), ("night", "Nighttime-only"), ("compound", "Compound")]
eff = pd.read_csv(SIM_RUN / "multiyear_siting_efficiency.csv")
eff = eff[eff.split.eq("main_2016-2019_to_2020-2023") & eff.cooling_mode.eq("tmax_tmin")
          & np.isclose(eff.delta_t_C, 1.0) & np.isclose(eff.budget_share, 0.2)]
g = eff.groupby(["heatwave_type", "strategy"])
sim = pd.DataFrame({"ratio_of_total_benefits": g.avoided_exposure.sum() / g.oracle_avoided.sum(),
                    "total_avoided_million_person_days_per_year": g.avoided_person_days_per_year.sum() / 1e6}).reset_index()
rows = []
TEX += ["", "% ===== Table S2: Siting rule & [Share (%) & Avoided] x Total, Daytime-only, Nighttime-only, Compound ====="]
for rule, label in RULES:
    row, cells = {"siting_rule": label}, [label]
    for heat, heat_label in TYPES:
        hit = sim[sim.strategy.eq(rule) & sim.heatwave_type.eq(heat)]
        require(len(hit) == 1, f"expected one row for {rule} / {heat}")
        share = 100 * float(hit.ratio_of_total_benefits.iloc[0])
        avoided = float(hit.total_avoided_million_person_days_per_year.iloc[0])
        row[f"{heat_label}_share_of_oracle_pct"] = round(share, 1)
        row[f"{heat_label}_avoided_million_person_days_per_year"] = round(avoided, 1)
        cells += [f"{share:.1f}", f"{avoided:,.0f}"]
    rows.append(row)
    if rule == "hwd_foresight":
        TEX.append(r"\addlinespace[3pt]")
    TEX.append(" & ".join(cells) + r" \\")
table_s2 = pd.DataFrame(rows)
table_s2.to_csv(OUT / "table_S2_siting_rules_by_heatwave_type.csv", index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# Table S3  Additivity among grid-cell-years with heatwaves (main definition)
# ------------------------------------------------------------
dml = pd.read_csv(DML_RUN / "dml_all_results.csv")
pos = dml[dml.scheme.eq("local_p90") & dml.w_variant.eq("base") & dml.analysis.eq("positive_ln")]
TREATMENTS = [("bld_coverage_z", "Building density"), ("bld_mean_h_m_z", "Mean building height")]
HEAT = [("day", "Daytime-only"), ("night", "Nighttime-only"), ("compound", "Compound")]


def pick(treatment, heat, outcome):
    hit = pos[pos.treatment.eq(treatment) & pos.heatwave_type.eq(heat) & pos.outcome.eq(outcome)]
    require(len(hit) == 1, f"expected one DML row for {treatment} / {heat} / {outcome}")
    return hit.iloc[0]


def with_ci(r):
    return minus(f"{r.theta:.3f} ({r.CI_lo_cluster:.3f}, {r.CI_hi_cluster:.3f})")


rows = []
TEX += ["", "% ===== Table S3: Morphology metric & Heatwave type & N & Exposure & Population & HWD & Sum & Difference ====="]
for treatment, treatment_label in TREATMENTS:
    for i, (heat, heat_label) in enumerate(HEAT):
        e, p, h = (pick(treatment, heat, o) for o in ["ln_exposure", "ln_population", "ln_hwd"])
        require(e.n == p.n == h.n, "sample sizes differ within a row")
        total = p.theta + h.theta
        rows.append({"morphology_metric": treatment_label, "heatwave_type": heat_label, "n": int(e.n),
                     "exposure": e.theta, "exposure_ci_lo": e.CI_lo_cluster, "exposure_ci_hi": e.CI_hi_cluster,
                     "population": p.theta, "population_ci_lo": p.CI_lo_cluster, "population_ci_hi": p.CI_hi_cluster,
                     "hwd": h.theta, "hwd_ci_lo": h.CI_lo_cluster, "hwd_ci_hi": h.CI_hi_cluster,
                     "sum_population_hwd": total, "difference": e.theta - total})
        TEX.append(f"{treatment_label if i == 0 else ''} & {heat_label} & {int(e.n):,} & {with_ci(e)} & {with_ci(p)} & "
                   f"{with_ci(h)} & {minus(f'{total:.3f}')} & {minus(f'{e.theta - total:.3f}')} \\\\")
    TEX.append(r"\addlinespace[3pt]")
table_s3 = pd.DataFrame(rows)
table_s3.round(4).to_csv(OUT / "table_S3_additivity_heatwave_cell_years.csv", index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# Table S4  Cross-validated performance of the exposure-prediction models
# ------------------------------------------------------------
cv_file = SHAP_BASE / "tables" / "ml_cv_metrics_xgboost.csv"
require(cv_file.is_file(), f"{cv_file} not found")
cv = pd.read_csv(cv_file).set_index("outcome")
rows = []
TEX += ["", "% ===== Table S4: Outcome & R2 (grouped) & RMSE (grouped) & R2 (random) & RMSE (random); mean (s.d.) across five folds ====="]
for outcome, label in [("log_exp_day", "Daytime-only exposure"), ("log_exp_night", "Nighttime-only exposure"),
                       ("log_exp_compound", "Compound exposure")]:
    r = cv.loc[outcome]
    rows.append({"outcome": label, "grouped_R2": r.cellkf_R2, "grouped_R2_sd": r.cellkf_R2_std, "grouped_RMSE": r.cellkf_RMSE,
                 "grouped_RMSE_sd": r.cellkf_RMSE_std, "random_R2": r.kf_R2, "random_R2_sd": r.kf_R2_std,
                 "random_RMSE": r.kf_RMSE, "random_RMSE_sd": r.kf_RMSE_std})
    TEX.append(f"{label} & {r.cellkf_R2:.2f} ({r.cellkf_R2_std:.2f}) & {r.cellkf_RMSE:.2f} ({r.cellkf_RMSE_std:.2f}) & "
               f"{r.kf_R2:.2f} ({r.kf_R2_std:.2f}) & {r.kf_RMSE:.2f} ({r.kf_RMSE_std:.2f}) \\\\")
table_s4 = pd.DataFrame(rows)
table_s4.round(4).to_csv(OUT / "table_S4_shap_cross_validation.csv", index=False, encoding="utf-8-sig")

(OUT / "SI_tables_rows.tex").write_text("\n".join(TEX) + "\n", encoding="utf-8")

print("Output:", OUT)